# 03 — Pipeline Blocks and a Custom Block

Phionyx Core executes turns through a canonical 46-block pipeline (contract
v3.8.0). Each block has a stable id, a determinism class (`strict`,
`seeded`, or `noisy_sensor`), and may declare patent-claim references.

This notebook lists the canonical order, then implements a tiny custom
block to show how to extend the pipeline.

**Public API used:** `PipelineBlock`, `BlockContext`, `BlockResult`,
`get_canonical_blocks`.


## 1. Imports

In [1]:
import asyncio
import hashlib

from phionyx_core import PipelineBlock, BlockContext, BlockResult
from phionyx_core.contracts.telemetry import get_canonical_blocks

print("PipelineBlock loaded. Default determinism:", PipelineBlock.determinism)


PipelineBlock loaded. Default determinism: strict


## 2. The canonical 46-block order

The pipeline contract is JSON shipped inside the package. Each version is
loadable separately for backwards compatibility.


In [2]:
blocks = get_canonical_blocks()  # defaults to v3.8.0
print(f"canonical blocks: {len(blocks)}")
print("first 10:")
for i, b in enumerate(blocks[:10], start=1):
    print(f"  {i:>2}. {b}")
print("...")
print("last 5:")
for i, b in enumerate(blocks[-5:], start=len(blocks) - 4):
    print(f"  {i:>2}. {b}")


canonical blocks: 46
first 10:
   1. kill_switch_gate
   2. time_update_sot
   3. input_safety_gate
   4. intent_classification
   5. context_retrieval_rag
   6. perceptual_frame_emit
   7. create_scenario_frame
   8. initialize_unified_state
   9. goal_evaluation
  10. goal_decomposition
...
last 5:
  42. response_build
  43. memory_consolidation
  44. audit_layer
  45. outcome_feedback
  46. learning_gate


## 3. Implementing a custom block

To plug into the pipeline a block subclasses `PipelineBlock` and
implements an async `execute(self, context) -> BlockResult`.

This `InputHashBlock` is intentionally trivial: it hashes the user input
and writes the digest into the result data. It is `strict`-deterministic
— same input always produces the same hash.


In [3]:
class InputHashBlock(PipelineBlock):
    """Compute SHA-256 of the user input. Strict-deterministic."""

    determinism = "strict"

    def __init__(self):
        super().__init__(block_id="example_input_hash")

    async def execute(self, context: BlockContext) -> BlockResult:
        digest = hashlib.sha256(context.user_input.encode("utf-8")).hexdigest()
        return BlockResult(
            block_id=self.block_id,
            status="ok",
            data={"input_sha256": digest, "length": len(context.user_input)},
        )

    def get_dependencies(self):
        return []  # no upstream dependencies for this example


block = InputHashBlock()
print(f"block_id:    {block.block_id}")
print(f"determinism: {block.determinism}")
print(f"deps:        {block.get_dependencies()}")


block_id:    example_input_hash
determinism: strict
deps:        []


## 4. Running the block

`BlockContext` is a dataclass — most fields are optional and default to
sensible values. We construct a minimal context and `await execute`.


In [4]:
ctx = BlockContext(
    user_input="the quick brown fox jumps over the lazy dog",
    card_type="example",
    card_title="determinism demo",
    scene_context="",
    card_result="",
)

result = await block.execute(ctx)  # Jupyter supports top-level await

print(f"block_id:  {result.block_id}")
print(f"status:    {result.status}")
print(f"is_success:{result.is_success()}")
print(f"data:      {result.data}")


block_id:  example_input_hash
status:    ok
is_success:True
data:      {'input_sha256': '05c6e08f1d9fdafa03147fcb8f82f124c76d2f70e3d989dc8aadb5e7d7450bec', 'length': 43}


## 5. Determinism check on the custom block

Run the block 100 times against the same context. The result data must
be identical every run — `strict` means strict.


In [5]:
hashes = set()

for _ in range(100):
    r = await block.execute(ctx)  # await — already inside Jupyter's event loop
    hashes.add(r.data["input_sha256"])

print(f"unique digests over 100 runs: {len(hashes)}")
print(f"digest: {next(iter(hashes))}")
assert len(hashes) == 1, "strict block must be deterministic"


unique digests over 100 runs: 1
digest: 05c6e08f1d9fdafa03147fcb8f82f124c76d2f70e3d989dc8aadb5e7d7450bec


## 6. Where blocks live in the canonical order

A real block would slot into the canonical order at a defined position.
For example, `input_safety_gate` runs at position 3 (so user input is
validated before it reaches anything else).


In [6]:
interesting = [
    "kill_switch_gate",            # safety, runs first
    "input_safety_gate",           # input validation
    "knowledge_boundary_check",    # epistemic gate
    "trust_evaluation",            # meta-cognitive trust
    "deliberative_ethics_gate",    # 4-framework ethics
    "behavioral_drift_detection",  # sustained drift signal
    "phi_computation",             # canonical phi
    "audit_layer",                 # tamper-evident audit
]

for name in interesting:
    idx = blocks.index(name)
    print(f"  #{idx+1:>2}  {name}")


  # 1  kill_switch_gate
  # 3  input_safety_gate
  #15  knowledge_boundary_check
  #16  trust_evaluation
  #18  deliberative_ethics_gate
  #23  behavioral_drift_detection
  #37  phi_computation
  #44  audit_layer


## What this proves

- The pipeline contract is data, not code — versioned and inspectable.
- Adding a block is a class with one async method.
- Determinism is declared on the class and verifiable in 100 lines.
- A real production block additionally emits an `AuditRecord` and
  declares its patent-claim refs; everything else is the same shape.

That is the whole extension model. No magic, no DI container, no plugin
manifest.
